# Declare a new run

Bare-minimum template for starting a new run: define its sources, its
tokenizer (reused if one already matches), a dataset and a pretraining config
under a fresh `run_id`, then check and declare it against the volume.

Copy this notebook per run and change `RUN_ID` and the knobs below.
Everything past declaring -- running jobs, loading artifacts back,
visualizing a plan, conflicts -- is `demo.ipynb`, not repeated here.

In [1]:
from artifact import (
    DataSet,
    ModelParameters,
    Pretraining,
    PretrainingConfig,
    Resources,
    Source,
    Tokenizer,
)


## Sources

In [5]:
odyssey = Source(
    name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
)
mobydick = Source(
    name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
)
romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)
montecristo = Source(
    name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt"
)
pride = Source(
    name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
)
frankenstein = Source(
    name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt"
)
greatexpectations = Source(
    name="greatexpectations", url="https://www.gutenberg.org/cache/epub/1400/pg1400.txt"
)
dracula = Source(
    name="dracula", url="https://www.gutenberg.org/cache/epub/345/pg345.txt"
)


## The run

In [6]:
RUN_ID = "RUN2"  # <- change this per run

# same tokenizer any other run trained on these sources would build -- if one's
# already declared, it's reused rather than rebuilt from scratch
tokenizer = Tokenizer(
    vocab_size=2200,
    kind="bpe",
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick),
)

dataset = DataSet.from_sources(
    run_id=RUN_ID,
    tokenizer=tokenizer,
    train_sources=[dracula, frankenstein],
    valid_sources=[pride,odyssey],
)

model_parameters = ModelParameters(hidden_size=64, num_layers=2)
config = PretrainingConfig(
    total_steps=2000, batch_size=32, lr=1e-3, seed=1, checkpoint_every=500
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=dataset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    config=config,
    allocated_resources=Resources(gpu_type='T4', gpu_count=1)
)

pretraining  # parameters all the way down; `commit` is hidden from the repr

Pretraining(run_id='RUN2', dataset=DataSet(run_id='RUN2', train_set=(TokenizedSource(tokenizer=Tokenizer(vocab_size=2200, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt')), kind='bpe'), source=Source(name='dracula', url='https://www.gutenberg.org/cache/epub/345/pg345.txt')), TokenizedSource(tokenizer=Tokenizer(vocab_size=2200, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt')), kind='bpe'), source=Source(name='frankenstein', url='https://www.gutenberg.org/cache/epub/84/pg84.txt'))), valid_set=(TokenizedSource(tokenizer=Tokenizer(vocab_size=2200, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.tx

## Declare

`declare_fn.remote(pretraining)` resolves the request and reconciles it
against the volume without writing anything -- everything shared (sources,
and the tokenizer if it matches one already declared) should read `done`;
everything new to this run should read `new`. `write=True` declares
whatever's `new`, refusing outright if anything's inconsistent.

In [7]:
import modal

from config import APP_NAME

declare_fn = modal.Function.from_name(APP_NAME, "declare")

In [8]:
print(declare_fn.remote(pretraining))  # preview, no writes

run RUN2 under /storage
  done       sources/odyssey (drift)
  done       sources/mobydick (drift)
  new        tokenizers/bpe-2.2k-e4649eb4ff
  new        sources/dracula
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/dracula
  new        sources/frankenstein
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/frankenstein
  new        sources/pride
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/pride
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/odyssey
  new        runs/RUN2/dataset
  new        runs/RUN2/pretraining

2 done, 10 new, 2 at another commit
ok -- 10 to declare


In [9]:
print(declare_fn.remote(pretraining, write=True))  # commits to the volume

run RUN2 under /storage
  done       sources/odyssey (drift)
  done       sources/mobydick (drift)
  new        tokenizers/bpe-2.2k-e4649eb4ff
  new        sources/dracula
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/dracula
  new        sources/frankenstein
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/frankenstein
  new        sources/pride
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/pride
  new        tokenizers/bpe-2.2k-e4649eb4ff/bin/odyssey
  new        runs/RUN2/dataset
  new        runs/RUN2/pretraining

2 done, 10 new, 2 at another commit
ok -- 10 to declare


Declared, not yet produced -- `declared` rows have a manifest but no files
yet. To actually run the jobs, see `demo.ipynb`'s job-list and `Job.run()`
cells, or drive `job_list(resolve(pretraining))` by hand.

In [ ]:
pretraining.manifest()